# homework

## Preparing the dataset

In [151]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

In [152]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv'

In [153]:
df = pd.read_csv("car_fuel_efficiency.csv")

In [154]:
df.columns = df.columns.str.lower()

In [155]:
df.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


In [156]:
df.describe().round()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,num_doors,fuel_efficiency_mpg
count,9704.0,9222.0,8996.0,9704.0,8774.0,9704.0,9202.0,9704.0
mean,200.0,4.0,150.0,3001.0,15.0,2011.0,-0.0,15.0
std,49.0,2.0,30.0,498.0,3.0,7.0,1.0,3.0
min,10.0,0.0,37.0,953.0,6.0,2000.0,-4.0,6.0
25%,170.0,3.0,130.0,2666.0,13.0,2006.0,-1.0,13.0
50%,200.0,4.0,149.0,2993.0,15.0,2012.0,0.0,15.0
75%,230.0,5.0,170.0,3335.0,17.0,2017.0,1.0,17.0
max,380.0,13.0,271.0,4739.0,24.0,2023.0,4.0,26.0


In [157]:
df.isnull().sum() 

engine_displacement      0
num_cylinders          482
horsepower             708
vehicle_weight           0
acceleration           930
model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors              502
fuel_efficiency_mpg      0
dtype: int64

In [158]:
numerical = list(df.select_dtypes(include=['int64', 'float64']).columns)
categorical = list(df.select_dtypes(include=['object', 'string']).columns)
numerical, categorical

(['engine_displacement',
  'num_cylinders',
  'horsepower',
  'vehicle_weight',
  'acceleration',
  'model_year',
  'num_doors',
  'fuel_efficiency_mpg'],
 ['origin', 'fuel_type', 'drivetrain'])

In [159]:
for c in categorical:
    df[c] = df[c].fillna('NA')

for n in numerical:
    df[n] = df[n].fillna(0.0)

In [160]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [161]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [162]:
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

In [163]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [164]:
df_train

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors
0,120,5.0,169.0,2966.679505,13.9,2005,USA,Gasoline,Front-wheel drive,-1.0
1,200,3.0,143.0,2950.822121,17.1,2013,Asia,Diesel,Front-wheel drive,-1.0
2,180,6.0,180.0,3078.221669,17.4,2007,USA,Gasoline,All-wheel drive,0.0
3,280,5.0,174.0,2797.991793,0.0,2016,USA,Diesel,All-wheel drive,0.0
4,250,4.0,133.0,2362.426930,16.3,2010,USA,Diesel,Front-wheel drive,-1.0
...,...,...,...,...,...,...,...,...,...,...
5817,230,3.0,176.0,3430.993044,17.9,2022,Europe,Diesel,All-wheel drive,0.0
5818,250,4.0,180.0,3067.664350,15.7,2010,Asia,Diesel,All-wheel drive,-1.0
5819,230,2.0,182.0,3041.964593,16.7,2010,Europe,Diesel,All-wheel drive,0.0
5820,180,7.0,147.0,2453.341430,15.2,2015,Europe,Gasoline,All-wheel drive,0.0


In [165]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.tree import export_text
from sklearn.metrics import root_mean_squared_error

In [166]:
train_dicts = df_train.fillna(0).to_dict(orient='records')

In [167]:
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)

## Question 1

In [168]:
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",1
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",1
,"max_leaf_

In [169]:
val_dicts = df_val.fillna(0).to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [170]:
y_pred = dt.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
print(rmse)

1.6104639028827594


In [171]:
y_pred = dt.predict(X_train)
rmse = root_mean_squared_error(y_train, y_pred)
print(rmse)

1.5864186811570804


In [172]:
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- vehicle_weight <= 3022.11
|   |--- value: [16.88]
|--- vehicle_weight >  3022.11
|   |--- value: [12.94]



## Question 2

In [173]:
from sklearn.ensemble import RandomForestRegressor

In [174]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
print(rmse)

0.4595777223092726


## Question 3

In [175]:
n_estimator_array = list(range(10, 201, 10))
for n in n_estimator_array:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    print(f'{n} trees: {round(rmse, 4)}')

10 trees: 0.4596
20 trees: 0.4536
30 trees: 0.4517
40 trees: 0.4487
50 trees: 0.4467
60 trees: 0.4455
70 trees: 0.4451
80 trees: 0.445
90 trees: 0.4449
100 trees: 0.4447
110 trees: 0.4436
120 trees: 0.4439
130 trees: 0.4437
140 trees: 0.4434
150 trees: 0.4429
160 trees: 0.4428
170 trees: 0.4428
180 trees: 0.4424
190 trees: 0.4425
200 trees: 0.4425


## Question 4

In [176]:
n_estimator_array = list(range(10, 201, 10))
for n in n_estimator_array:
    for d in [10, 15, 20, 25]:
        rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1, max_depth=d)
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        print(f'{n} trees, {d} depth: {round(rmse, 4)}')

10 trees, 10 depth: 0.4502
10 trees, 15 depth: 0.4576
10 trees, 20 depth: 0.4587
10 trees, 25 depth: 0.4595
20 trees, 10 depth: 0.4469
20 trees, 15 depth: 0.4531
20 trees, 20 depth: 0.454
20 trees, 25 depth: 0.4539
30 trees, 10 depth: 0.4455
30 trees, 15 depth: 0.4509
30 trees, 20 depth: 0.452
30 trees, 25 depth: 0.4521
40 trees, 10 depth: 0.4431
40 trees, 15 depth: 0.4486
40 trees, 20 depth: 0.4493
40 trees, 25 depth: 0.449
50 trees, 10 depth: 0.442
50 trees, 15 depth: 0.4463
50 trees, 20 depth: 0.4473
50 trees, 25 depth: 0.4469
60 trees, 10 depth: 0.4417
60 trees, 15 depth: 0.4453
60 trees, 20 depth: 0.4461
60 trees, 25 depth: 0.4457
70 trees, 10 depth: 0.4413
70 trees, 15 depth: 0.4446
70 trees, 20 depth: 0.4455
70 trees, 25 depth: 0.4453
80 trees, 10 depth: 0.4414
80 trees, 15 depth: 0.4446
80 trees, 20 depth: 0.4458
80 trees, 25 depth: 0.4452
90 trees, 10 depth: 0.4415
90 trees, 15 depth: 0.4446
90 trees, 20 depth: 0.4458
90 trees, 25 depth: 0.445
100 trees, 10 depth: 0.4412
100 t

## Question 5

In [177]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1, max_depth=20)
rf.fit(X_train, y_train)

feature_importance = pd.Series(
      rf.feature_importances_,
      index=dv.get_feature_names_out()
  ).sort_values(ascending=False)

print(feature_importance)

vehicle_weight                  0.959150
horsepower                      0.015998
acceleration                    0.011480
engine_displacement             0.003273
model_year                      0.003212
num_cylinders                   0.002343
num_doors                       0.001635
origin=USA                      0.000540
origin=Europe                   0.000519
origin=Asia                     0.000462
fuel_type=Gasoline              0.000360
drivetrain=All-wheel drive      0.000357
drivetrain=Front-wheel drive    0.000345
fuel_type=Diesel                0.000325
dtype: float64


## Question 6

In [178]:
import xgboost as xgb

In [179]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

In [180]:
watchlist = [(dtrain, 'train'), (dval, 'val')]

In [181]:
%%capture output

xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=10)

In [182]:
s = output.stdout
s.splitlines()

['[0]\ttrain-rmse:1.81393\tval-rmse:1.85444',
 '[10]\ttrain-rmse:0.37115\tval-rmse:0.43896',
 '[20]\ttrain-rmse:0.33553\tval-rmse:0.43376',
 '[30]\ttrain-rmse:0.31475\tval-rmse:0.43752',
 '[40]\ttrain-rmse:0.30202\tval-rmse:0.43968',
 '[50]\ttrain-rmse:0.28456\tval-rmse:0.44140',
 '[60]\ttrain-rmse:0.26768\tval-rmse:0.44290',
 '[70]\ttrain-rmse:0.25489\tval-rmse:0.44531',
 '[80]\ttrain-rmse:0.24254\tval-rmse:0.44689',
 '[90]\ttrain-rmse:0.23193\tval-rmse:0.44839',
 '[99]\ttrain-rmse:0.21950\tval-rmse:0.45018']

In [183]:
y_pred = model.predict(dval)
rmse = root_mean_squared_error(y_val, y_pred)
print(f'{round(rmse, 4)}')

0.4502


In [184]:
%%capture output

xgb_params = {
    'eta': 0.1, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=10)

In [185]:
s = output.stdout
s.splitlines()

['[0]\ttrain-rmse:2.28944\tval-rmse:2.34561',
 '[10]\ttrain-rmse:0.91008\tval-rmse:0.94062',
 '[20]\ttrain-rmse:0.48983\tval-rmse:0.53064',
 '[30]\ttrain-rmse:0.38342\tval-rmse:0.44289',
 '[40]\ttrain-rmse:0.35343\tval-rmse:0.42746',
 '[50]\ttrain-rmse:0.33998\tval-rmse:0.42498',
 '[60]\ttrain-rmse:0.33054\tval-rmse:0.42456',
 '[70]\ttrain-rmse:0.32202\tval-rmse:0.42503',
 '[80]\ttrain-rmse:0.31667\tval-rmse:0.42563',
 '[90]\ttrain-rmse:0.31059\tval-rmse:0.42586',
 '[99]\ttrain-rmse:0.30419\tval-rmse:0.42623']

In [146]:
y_pred = model.predict(dval)
rmse = root_mean_squared_error(y_val, y_pred)
print(f'{round(rmse, 4)}')

0.4262


In [186]:
def train_xgb(eta):
      params = {
          'eta': eta,
          'max_depth': 6,
          'min_child_weight': 1,
          'objective': 'reg:squarederror',
          'nthread': 8,
          'seed': 1,
          'verbosity': 1,
      }

      results = {}

      model = xgb.train(
          params,
          dtrain,
          num_boost_round=100,
          evals=watchlist,
          evals_result=results,
          verbose_eval=False
      )

      return model, results

for eta in [0.3, 0.1]:
    model, results = train_xgb(eta)

    print(
        eta,
        'final:', results['val']['rmse'][-1],
        'best:', min(results['val']['rmse'])
    )

0.3 final: 0.45017754981610764 best: 0.4334861295405598
0.1 final: 0.4262280017058715 best: 0.4242625629140815
